# Cross-Dataset Analysis: Attention Sinks in Graph Transformers

This notebook loads trained models and precomputed diagnostics from all experiments,
then produces the main analysis figures for the report:

1. **Context Size Sweep** (Barbero Fig 5 analogue) — sink metrics vs graph size
2. **Layer-wise diagnostics** per dataset — 6-panel trajectory plots
3. **VNode vs No-VNode comparison** per dataset
4. **Cross-experiment summary table**

Training was done via `scripts/run_experiment.py` on EC2. This notebook is analysis-only.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import yaml
import pickle
from collections import OrderedDict

matplotlib.rcParams.update({
    'font.size': 11,
    'figure.dpi': 150,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

from src.diagnostics import load_diagnostics, aggregate_metrics, print_summary

OUTPUTS_DIR = '../outputs'
FIGURES_DIR = '../outputs/figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

print('Setup complete.')

## 1. Load all experiments

In [ ]:
# Define the full experiment grid
EXPERIMENTS = OrderedDict([
    # Context size sweep: VNode vs no-VNode across 4 datasets
    ('zinc-vnode',        {'dataset': 'ZINC',      'avg_nodes': 23,  'vnode': True,  'group': 'context'}),
    ('zinc-novnode',      {'dataset': 'ZINC',      'avg_nodes': 23,  'vnode': False, 'group': 'context'}),
    ('mnist-vnode',       {'dataset': 'MNIST-SP',  'avg_nodes': 70,  'vnode': True,  'group': 'context'}),
    ('mnist-novnode',     {'dataset': 'MNIST-SP',  'avg_nodes': 70,  'vnode': False, 'group': 'context'}),
    ('peptides-vnode',    {'dataset': 'Peptides',  'avg_nodes': 150, 'vnode': True,  'group': 'context'}),
    ('peptides-novnode',  {'dataset': 'Peptides',  'avg_nodes': 150, 'vnode': False, 'group': 'context'}),
    ('pascal-vnode',      {'dataset': 'PascalVOC', 'avg_nodes': 479, 'vnode': True,  'group': 'context'}),
    ('pascal-novnode',    {'dataset': 'PascalVOC', 'avg_nodes': 479, 'vnode': False, 'group': 'context'}),
    # Depth ablation on PascalVOC no-VNode
    ('pascal-novnode-3L', {'dataset': 'PascalVOC', 'avg_nodes': 479, 'vnode': False, 'group': 'depth'}),
    ('pascal-novnode-8L', {'dataset': 'PascalVOC', 'avg_nodes': 479, 'vnode': False, 'group': 'depth'}),
])

# Load diagnostics and configs
results = {}
for eid, meta in EXPERIMENTS.items():
    diag_path = os.path.join(OUTPUTS_DIR, eid, 'diagnostics.pkl')
    config_path = os.path.join(OUTPUTS_DIR, eid, 'config.yaml')
    
    if not os.path.exists(diag_path):
        print(f'  MISSING: {eid}')
        continue
    
    with open(config_path) as f:
        config = yaml.safe_load(f)
    metrics = load_diagnostics(diag_path)
    agg, layers = aggregate_metrics(metrics)
    
    results[eid] = {
        'config': config,
        'metrics_raw': metrics,
        'agg': agg,
        'layers': layers,
        **meta,
    }
    print(f'  Loaded: {eid} ({len(layers)} layers, {len(metrics[0])} graphs)')

print(f'\nLoaded {len(results)}/{len(EXPERIMENTS)} experiments')

## 2. Cross-experiment summary table

In [ ]:
def extract_summary(r):
    """Extract key scalar metrics from an experiment's aggregated diagnostics."""
    agg = r['agg']
    cfg = r['config']
    summary = {
        'Dataset': r['dataset'],
        'Nodes': r['avg_nodes'],
        'VNode': 'ON' if r['vnode'] else 'OFF',
        'Layers': cfg['architecture']['num_layers'],
        'MPNN': cfg['architecture']['mpnn'],
        'PE': cfg['pe']['type'],
    }
    for key, label in [('max_sink_score', 'Max Sink'), ('overall_sink_rate', 'Sink Rate'),
                        ('max_to_mean_ratio', 'Norm Ratio'), ('matrix_entropy', 'Min Entropy'),
                        ('anisotropy', 'Max Aniso'), ('mixing_score', 'Min Mixing')]:
        if key in agg:
            arr = agg[key]['mean']
            if key == 'matrix_entropy':
                summary[label] = np.nanmin(arr)
            elif key == 'mixing_score':
                summary[label] = np.nanmin(arr)
            else:
                summary[label] = np.nanmax(arr)
        else:
            summary[label] = np.nan
    return summary


# Print table
header = f"{'Experiment':<22} {'Dataset':<10} {'N':>4} {'VN':>3} {'L':>2} {'MPNN':<8} {'MaxSink':>8} {'SinkR':>6} {'NormR':>6} {'MinEnt':>7} {'MaxAni':>7}"
print(header)
print('=' * len(header))

summaries = {}
for eid, r in results.items():
    s = extract_summary(r)
    summaries[eid] = s
    print(f"{eid:<22} {s['Dataset']:<10} {s['Nodes']:>4} {s['VNode']:>3} {s['Layers']:>2} {s['MPNN']:<8} "
          f"{s['Max Sink']:>8.4f} {s['Sink Rate']:>6.3f} {s['Norm Ratio']:>6.2f} {s['Min Entropy']:>7.4f} {s['Max Aniso']:>7.4f}")

## 3. Main Result: Context Size Sweep (Barbero Fig 5 analogue)

**Key question:** Does sink strength increase with graph size (number of nodes),
analogous to how sink strength increases with context length in LLMs?

We plot sink metrics against average number of nodes per dataset,
with separate lines for VNode ON vs OFF.

In [ ]:
# Collect context sweep data (exclude depth ablation)
ctx_results = {eid: r for eid, r in results.items() if r['group'] == 'context'}

datasets_ordered = ['ZINC', 'MNIST-SP', 'Peptides', 'PascalVOC']
avg_nodes = [23, 70, 150, 479]

def get_metric_by_dataset(metric_name, vnode, reducer='max'):
    """Get metric value for each dataset."""
    values = []
    for ds in datasets_ordered:
        val = np.nan
        for eid, r in ctx_results.items():
            if r['dataset'] == ds and r['vnode'] == vnode:
                if metric_name in r['agg']:
                    arr = r['agg'][metric_name]['mean']
                    if reducer == 'max':
                        val = np.nanmax(arr)
                    elif reducer == 'min':
                        val = np.nanmin(arr)
                    elif reducer == 'last':
                        valid = arr[~np.isnan(arr)]
                        val = valid[-1] if len(valid) > 0 else np.nan
        values.append(val)
    return np.array(values)


fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Sink Metrics vs Context Size (Number of Nodes)', fontsize=14, fontweight='bold')

metrics_config = [
    ('max_sink_score', 'Max Sink Score', 'max'),
    ('overall_sink_rate', 'Overall Sink Rate', 'max'),
    ('max_to_mean_ratio', 'Max/Mean Norm Ratio', 'max'),
    ('matrix_entropy', 'Min Matrix Entropy', 'min'),
    ('anisotropy', 'Max Anisotropy $p_1$', 'max'),
    ('dirichlet_energy', 'Final Dirichlet Energy', 'last'),
]

for ax, (metric, title, reducer) in zip(axes.flat, metrics_config):
    for vnode, label, color, marker in [(True, 'With VNode', 'tab:red', 's'),
                                        (False, 'Without VNode', 'tab:blue', 'o')]:
        vals = get_metric_by_dataset(metric, vnode, reducer)
        valid = ~np.isnan(vals)
        ax.plot(np.array(avg_nodes)[valid], vals[valid], color=color, linewidth=2,
                marker=marker, markersize=8, label=label)
    
    ax.set_xlabel('Avg nodes per graph')
    ax.set_xscale('log')
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    # Add dataset labels on x-axis
    ax.set_xticks(avg_nodes)
    ax.set_xticklabels(['ZINC\n(23)', 'MNIST\n(70)', 'Pept.\n(150)', 'Pascal\n(479)'], fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig1_context_size_sweep.pdf'), bbox_inches='tight', dpi=150)
plt.show()
print('Saved: fig1_context_size_sweep.pdf')

## 4. Layer-wise diagnostics per dataset

For each of the 4 datasets, plot the 6-panel layer-wise trajectory,
overlaying VNode vs no-VNode.

In [ ]:
metric_panels = [
    ('max_sink_score', 'Max Sink Score'),
    ('overall_sink_rate', 'Sink Rate'),
    ('matrix_entropy', 'Matrix Entropy $H(X)$'),
    ('anisotropy', 'Anisotropy $p_1$'),
    ('dirichlet_energy', 'Dirichlet Energy'),
    ('max_to_mean_ratio', 'Norm Ratio $\\|h_{max}\\|^2 / \\overline{\\|h\\|}^2$'),
]

for ds_name in datasets_ordered:
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle(f'Layer-wise Diagnostics — {ds_name}', fontsize=14, fontweight='bold')
    
    for ax, (metric_name, title) in zip(axes.flat, metric_panels):
        for eid, r in ctx_results.items():
            if r['dataset'] != ds_name:
                continue
            color = 'tab:red' if r['vnode'] else 'tab:blue'
            label = 'VNode' if r['vnode'] else 'No VNode'
            
            if metric_name in r['agg']:
                mean = r['agg'][metric_name]['mean']
                std = r['agg'][metric_name]['std']
                layers = np.array(r['layers'])
                valid = ~np.isnan(mean)
                ax.plot(layers[valid], mean[valid], color=color, linewidth=2,
                        marker='o', markersize=3, label=label)
                ax.fill_between(layers[valid], (mean - std)[valid], (mean + std)[valid],
                               alpha=0.15, color=color)
        
        ax.set_xlabel('Layer')
        ax.set_title(title)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    safe_name = ds_name.lower().replace('-', '_').replace(' ', '_')
    plt.savefig(os.path.join(FIGURES_DIR, f'fig2_layers_{safe_name}.pdf'), bbox_inches='tight', dpi=150)
    plt.show()
    print(f'Saved: fig2_layers_{safe_name}.pdf')

## 5. VNode effect: does VNode absorb the sink?

Compare the difference in sink strength between VNode and no-VNode conditions
across all datasets. If VNode acts as an attention sink, adding it should
concentrate attention more (higher sink score) because the VNode provides a
structurally unique target.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

for ax, ds_name in zip(axes, datasets_ordered):
    for eid, r in ctx_results.items():
        if r['dataset'] != ds_name:
            continue
        color = 'tab:red' if r['vnode'] else 'tab:blue'
        label = 'VNode' if r['vnode'] else 'No VNode'
        
        if 'max_sink_score' in r['agg']:
            mean = r['agg']['max_sink_score']['mean']
            std = r['agg']['max_sink_score']['std']
            layers = np.array(r['layers'])
            valid = ~np.isnan(mean)
            ax.plot(layers[valid], mean[valid], color=color, linewidth=2,
                    marker='o', markersize=4, label=label)
            ax.fill_between(layers[valid], (mean - std)[valid], (mean + std)[valid],
                           alpha=0.15, color=color)
    
    # Add uniform attention baseline
    n = [r for r in ctx_results.values() if r['dataset'] == ds_name][0]['avg_nodes']
    ax.axhline(1.0 / n, color='gray', linestyle='--', alpha=0.5, label=f'Uniform (1/{n})')
    
    ax.set_xlabel('Layer')
    ax.set_ylabel('Max Sink Score')
    ax.set_title(f'{ds_name} (~{n} nodes)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('VNode Effect on Sink Score Across Datasets', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig3_vnode_effect.pdf'), bbox_inches='tight', dpi=150)
plt.show()

## 6. Compression analysis: do sinks correlate with entropy collapse?

Queipo-de-Llano et al. predict that if a node develops massive activations,
matrix entropy should drop (compression valley) and anisotropy should rise.
We check if this co-occurrence holds across datasets.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Compression Diagnostics: Entropy (top) and Anisotropy (bottom)', fontsize=14, fontweight='bold')

for col, ds_name in enumerate(datasets_ordered):
    for row, (metric, ylabel) in enumerate([('matrix_entropy', 'Matrix Entropy $H(X)$'),
                                             ('anisotropy', 'Anisotropy $p_1$')]):
        ax = axes[row, col]
        for eid, r in ctx_results.items():
            if r['dataset'] != ds_name:
                continue
            color = 'tab:red' if r['vnode'] else 'tab:blue'
            label = 'VNode' if r['vnode'] else 'No VNode'
            
            if metric in r['agg']:
                mean = r['agg'][metric]['mean']
                std = r['agg'][metric]['std']
                layers = np.array(r['layers'])
                valid = ~np.isnan(mean)
                ax.plot(layers[valid], mean[valid], color=color, linewidth=2,
                        marker='o', markersize=3, label=label)
                ax.fill_between(layers[valid], (mean - std)[valid], (mean + std)[valid],
                               alpha=0.15, color=color)
        
        ax.set_xlabel('Layer')
        if col == 0:
            ax.set_ylabel(ylabel)
        if row == 0:
            ax.set_title(ds_name)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig4_compression.pdf'), bbox_inches='tight', dpi=150)
plt.show()

## 7. Over-smoothing: Dirichlet energy across datasets

Dirichlet energy measures how different neighboring node representations are.
A monotonic decrease signals over-smoothing (Arroyo et al., 2025).
We check whether VNode changes the over-smoothing dynamics.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

for ax, ds_name in zip(axes, datasets_ordered):
    for eid, r in ctx_results.items():
        if r['dataset'] != ds_name:
            continue
        color = 'tab:red' if r['vnode'] else 'tab:blue'
        label = 'VNode' if r['vnode'] else 'No VNode'
        
        if 'dirichlet_energy' in r['agg']:
            mean = r['agg']['dirichlet_energy']['mean']
            layers = np.array(r['layers'])
            valid = ~np.isnan(mean)
            # Normalize by first layer for comparability
            if mean[valid][0] > 0:
                normalized = mean[valid] / mean[valid][0]
                ax.plot(layers[valid], normalized, color=color, linewidth=2,
                        marker='o', markersize=3, label=label)
    
    ax.set_xlabel('Layer')
    ax.set_ylabel('Dirichlet Energy (normalized)')
    ax.set_title(ds_name)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Over-smoothing: Normalized Dirichlet Energy Across Layers', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig5_dirichlet.pdf'), bbox_inches='tight', dpi=150)
plt.show()

## 8. Mixing score: how diffuse is attention?

Mixing score = average Shannon entropy of attention distributions.
High = diffuse (every node attends equally), Low = concentrated (sharp attention).
We expect larger graphs to have lower mixing scores (more concentration needed).

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

for ax, ds_name in zip(axes, datasets_ordered):
    for eid, r in ctx_results.items():
        if r['dataset'] != ds_name:
            continue
        color = 'tab:red' if r['vnode'] else 'tab:blue'
        label = 'VNode' if r['vnode'] else 'No VNode'
        
        if 'mixing_score' in r['agg']:
            mean = r['agg']['mixing_score']['mean']
            std = r['agg']['mixing_score']['std']
            layers = np.array(r['layers'])
            valid = ~np.isnan(mean)
            ax.plot(layers[valid], mean[valid], color=color, linewidth=2,
                    marker='o', markersize=3, label=label)
            ax.fill_between(layers[valid], (mean - std)[valid], (mean + std)[valid],
                           alpha=0.15, color=color)
    
    # Reference: maximum entropy (uniform attention)
    n = [r for r in ctx_results.values() if r['dataset'] == ds_name][0]['avg_nodes']
    ax.axhline(np.log(n), color='gray', linestyle='--', alpha=0.5, label=f'$\\ln({n})$ (uniform)')
    
    ax.set_xlabel('Layer')
    ax.set_ylabel('Mixing Score')
    ax.set_title(ds_name)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Mixing Score (Attention Entropy) Across Layers', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig6_mixing_score.pdf'), bbox_inches='tight', dpi=150)
plt.show()

## 9. Key findings summary

In [ ]:
print('=' * 70)
print('KEY FINDINGS')
print('=' * 70)

print('\n--- Context Size Effect (H1: Barbero Fig 5 analogue) ---')
for vnode in [True, False]:
    label = 'VNode' if vnode else 'No VNode'
    vals = get_metric_by_dataset('max_sink_score', vnode, 'max')
    print(f'  {label} max sink score:  ', 
          '  '.join(f'{ds}: {v:.4f}' for ds, v in zip(datasets_ordered, vals)))

print('\n--- VNode Effect (H4) ---')
for ds in datasets_ordered:
    vn_val = np.nan
    novn_val = np.nan
    for eid, r in ctx_results.items():
        if r['dataset'] != ds:
            continue
        if 'max_sink_score' in r['agg']:
            val = np.nanmax(r['agg']['max_sink_score']['mean'])
            if r['vnode']:
                vn_val = val
            else:
                novn_val = val
    diff = vn_val - novn_val
    print(f'  {ds:12s}: VNode={vn_val:.4f}, NoVNode={novn_val:.4f}, diff={diff:+.4f}')

print('\n--- Compression (H3: Queipo-de-Llano) ---')
for eid, r in results.items():
    if r['group'] != 'context':
        continue
    agg = r['agg']
    min_ent = np.nanmin(agg['matrix_entropy']['mean']) if 'matrix_entropy' in agg else np.nan
    max_aniso = np.nanmax(agg['anisotropy']['mean']) if 'anisotropy' in agg else np.nan
    print(f'  {eid:22s}: min_entropy={min_ent:.4f}, max_aniso={max_aniso:.4f}')

print('\n--- Over-smoothing ---')
for eid, r in results.items():
    if r['group'] != 'context':
        continue
    if 'dirichlet_energy' in r['agg']:
        de = r['agg']['dirichlet_energy']['mean']
        valid = de[~np.isnan(de)]
        ratio = valid[-1] / valid[0] if valid[0] > 0 else np.nan
        print(f'  {eid:22s}: DE ratio (last/first) = {ratio:.4f}')